In [ ]:
MODEL_PATH = os.path.join(project_root, 'slovene_pipeline', 'subtaskA_notebooks', 'finalized_model.sav')

if os.path.exists(MODEL_PATH):
    irony_classifier = joblib.load(MODEL_PATH)
    if isinstance(irony_classifier, dict) and "vectorizer" in irony_classifier:
        vectorizer = irony_classifier["vectorizer"]
        clf = irony_classifier["classifier"]
    else:
        clf = irony_classifier
        vectorizer = None 
else:
    print(f"Model not found at {MODEL_PATH}")

# Option 1: LLM Translator (Groq/OpenAI base)
API_KEY = "your-api-key-here" 
llm_translator = LLMSarcasmTranslator(api_key=API_KEY)

# Option 2: Local HuggingFace Translator (e.g., mT5 trained on dataset)
local_translator = T5SarcasmTranslator(model_name="csebuetnlp/mT5_multilingual_XLSum")
# Uncomment next line to load local weights if trained locally
# local_translator.load_model(os.path.join(project_root, "irony_translation", "from_small_dataset", "mT5_multilingual_XLSum"))

# Choose which one to pass to the pipeline
translator = llm_translator 
# translator = local_translator

In [ ]:
import sys
import os
import re
import joblib

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from slovene_pipeline.word2vec_api import Word2VecAPI, maybe_load_emoji2vec
from slovene_pipeline.features import build_tfidf, build_w2v_mean, combine_features
from irony_translation.LLMSarcasmTranslator import LLMSarcasmTranslator
from irony_translation.SarcasmTranslator import T5SarcasmTranslator

In [ ]:
def is_ironic(sentence: str, clf_model=None, vectorizer_model=None, w2v_model=None, emoji_model=None) -> bool:
    if clf_model is None:
        return False
        
    corpus = [sentence]
    try:
        tfidf_x, _ = build_tfidf(corpus, fitted_vectorizer=vectorizer_model)
        w2v_x = build_w2v_mean(corpus, w2v_model, emoji_model)
        x = combine_features(tfidf_x, w2v_x)
        
        preds = clf_model.predict(x)
        return bool(preds[0])
    except Exception as e:
        print(f"Error during prediction: {e}")
        return False


In [ ]:
def translate_ironic_sentence(sentence: str, translator_model) -> str:
    try:
        if hasattr(translator_model, 'zero_shot'):
            # Used by LLMSarcasmTranslator
            non_ironic = translator_model.zero_shot(sentence)
        elif hasattr(translator_model, 'generate'):
            # Used by T5SarcasmTranslator
            non_ironic = translator_model.generate(sentence)
        else:
            return sentence
            
        return non_ironic.strip()
    except Exception as e:
        print(f"Translation failed: {e}")
        return sentence

In [ ]:
def process_text_pipeline(input_text: str, clf_model=None, vectorizer_model=None, w2v_model=None, emoji_model=None, translator_model=None):
    is_ironic_text = is_ironic(
        input_text, 
        clf_model=clf_model, 
        vectorizer_model=vectorizer_model, 
        w2v_model=w2v_model, 
        emoji_model=emoji_model
    )
    
    final_text = input_text
    transformed = False
    
    if is_ironic_text and translator_model is not None:
        final_text = translate_ironic_sentence(input_text, translator_model)
        transformed = True
        
    result = {
        "original_text": input_text,
        "final_text": final_text,
        "is_ironic": is_ironic_text,
        "transformed": transformed
    }
    
    return result

Example usage:

In [ ]:
sample_texts = [
    "To je bil res odličen dan. Komaj čakam, da ponovimo!", # Expected not ironic
    "Oh, super, spet dežuje ravno ko grem na morje. To je bil res 'odličen' dan.", # Expected ironic
    "Danes sem šel v trgovino. Prodajalka je bila izjemno ustrežljiva, kar ni tipično, ja pa seveda.", # Mixed
]

for text in sample_texts:
    CLF = globals().get('clf', None)
    VEC = globals().get('vectorizer', None)
    W2V = globals().get('w2v', None)
    EMOJI = globals().get('emoji_model', None)
    TRANS = globals().get('translator', None)
    
    if CLF is None:
        def mock_is_ironic(sentence, *args, **kwargs):
            return "odličen" in sentence.lower() and "dežuje" in sentence.lower() or "seveda" in sentence.lower()
        
        original_is_ironic = is_ironic
        is_ironic = mock_is_ironic
    
    output = process_text_pipeline(text, clf_model=CLF, vectorizer_model=VEC, w2v_model=W2V, emoji_model=EMOJI, translator_model=TRANS)
    
    import json
    print(json.dumps(output, indent=2, ensure_ascii=False))
    
    if CLF is None:
         is_ironic = original_is_ironic

In [ ]:
import pandas as pd

def run_experiment_on_file(input_csv_path: str, output_csv_path: str):
    df = pd.read_csv(input_csv_path)
    
    CLF = globals().get('clf', None)
    VEC = globals().get('vectorizer', None)
    W2V = globals().get('w2v', None)
    EMOJI = globals().get('emoji_model', None)
    TRANS = globals().get('translator', None)
    
    results = []
    
    for index, row in df.iterrows():
        text = str(row['text'])
        true_label = row.get('true_label', '')
        
        output = process_text_pipeline(
            text, 
            clf_model=CLF, 
            vectorizer_model=VEC, 
            w2v_model=W2V, 
            emoji_model=EMOJI, 
            translator_model=TRANS
        )
        
        results.append({
            'input_text': text,
            'true_label': true_label,
            'predicted_ironic': output.get('is_ironic', False),
            'output_text': output.get('final_text', text),
            'TP_FP_TN_FN': '' # manual or whatever
        })
        
    out_df = pd.DataFrame(results)
    
    out_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    print(f"Eksperiment zaključen. Podatki shranjeni v: {output_csv_path}")

# usage
# run_experiment_on_file("ročni_tviti.csv", "rezultati_eksperimenta_normal.csv")